In [ ]:
try:
    import pyspark.sql.functions as F
    from pyspark.sql import Window
    from pyspark.sql.types import IntegerType
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

SCHEMA_ORIGEM = "bronze"
TABELA_ORIGEM = "tb_movies_info"
SCHEMA_DESTINO = "silver"
TABELA_DESTINO = "tb_info_filmes"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

In [ ]:
df_bronze = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_ORIGEM}")

colunas_esperadas = {
    "id", "tconst", "title", "original_title",
    "original_language", "release_date", "runtime",
    "status", "overview", "tagline", "ingestion_datetime"
}
colunas_ausentes = colunas_esperadas.difference(df_bronze.columns)
if colunas_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_ausentes)}")

# mantém a Bronze inalterada e aplica as transformações em uma cópia
df_tratado = (
    df_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("id_imdb", F.trim(F.col("tconst")))
    .withColumn("titulo", F.trim(F.col("title")))
    .withColumn("titulo_original", F.trim(F.col("original_title")))
    .withColumn("idioma_original", F.trim(F.col("original_language")))
    .withColumn("duracao_minutos", F.expr("try_cast(runtime AS INT)"))
    .withColumn("sinopse", F.trim(F.col("overview")))
    .withColumn("tagline", F.trim(F.col("tagline")))
)

In [ ]:
# normaliza caixa, espaços e hífens antes de traduzir os status
status_normalizado = F.lower(
    F.trim(F.regexp_replace(F.coalesce(F.col("status"), F.lit("")), r"[\s_-]+", " "))
)
df_tratado = df_tratado.withColumn("status_normalizado", status_normalizado)
df_tratado = df_tratado.withColumn(
    "status",
    F.when(F.col("status_normalizado") == "released", F.lit("Lançado"))
     .when(F.col("status_normalizado") == "post production", F.lit("Pós-Produção"))
     .when(F.col("status_normalizado") == "in production", F.lit("Em Produção"))
     .when(F.col("status_normalizado") == "planned", F.lit("Planejado"))
     .otherwise(F.lit("Não Informado"))
).drop("status_normalizado")

In [ ]:
# trata os formatos de data observados; valores inválidos viram NULL
data_texto = F.trim(F.col("release_date"))
data_lancamento = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
    F.expr("try_to_date(release_date, 'dd-MM-yyyy')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')")
)

df_tratado = (
    df_tratado
    .withColumn("data_lancamento", data_lancamento)
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")).cast(IntegerType()))
)

# mantém a versão mais recente de cada filme conforme a ingestão
janela_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_tratado = (
    df_tratado
    .withColumn("ordem_ingestao", F.row_number().over(janela_mais_recente))
    .where(F.col("id_filme").isNotNull() & (F.col("ordem_ingestao") == 1))
    .drop("ordem_ingestao", "id", "tconst", "title", "original_title",
          "original_language", "release_date", "runtime", "overview")
)

colunas_silver = [
    "id_filme", "id_imdb", "titulo", "titulo_original",
    "idioma_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "status", "sinopse", "tagline",
    "ingestion_datetime"
]
df_silver = df_tratado.select(*colunas_silver)

In [ ]:
# grava a Silver em overwrite para permitir reprocessamento idempotente
(df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}").limit(10))

In [ ]:
# valida unicidade, tipos tratados e datas não interpretáveis
df_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}")
duplicados = (
    df_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
datas_invalidas = (
    df_validacao.where(F.col("data_lancamento").isNull()).count()
)

if duplicados != 0:
    raise AssertionError(f"Há {duplicados} ids de filme duplicados na Silver.")

display(
    df_validacao.groupBy("status").count().orderBy(F.col("count").desc())
)
print(f"Registros Silver: {df_validacao.count()}")
print(f"Datas nulas (inválidas ou ausentes): {datas_invalidas}")
print(f"Duplicidades por id_filme: {duplicados}")